# Komaquillで画像を生成する

1. 有料Colabで計算ユニット残高を確認し、「ランタイムのタイプを変更」でGPUを選択します。
2. 左側の鍵アイコンから `KOMAQUILL_PASSWORD` を登録し、このノートブックからのアクセスを許可します。20文字以上のランダムなパスワードを使用します。
3. ローカルの `komaquill start` に表示されたコミットIDを下の欄へ貼り付けて実行します。
4. 表示された接続先URLをローカルへ入力します。Google Driveのマウントは不要です。
5. 利用後はローカルの回収状況を確認し、「ランタイムを接続解除して削除」を実行します。

ノートブックの出力は保存・共有しないでください。

In [ ]:
# @title ComfyUIを起動する
import os
import re
import subprocess
import sys
from pathlib import Path

from google.colab import userdata

source_revision = ""  # @param {type:"string"}

if not re.fullmatch(r"[0-9a-f]{40}", source_revision):
    raise ValueError("startに表示された40文字のコミットIDを指定してください。")
repo = Path("/content/komaquill/artifacts/source") / source_revision
if not repo.exists():
    repo.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "git",
            "clone",
            "--filter=blob:none",
            "--no-checkout",
            "https://github.com/satter0827/komaquill.git",
            str(repo),
        ],
        check=True,
    )
    subprocess.run(["git", "-C", str(repo), "checkout", "--detach", source_revision], check=True)
env = dict(os.environ)
env["KOMAQUILL_PASSWORD"] = userdata.get("KOMAQUILL_PASSWORD")
with subprocess.Popen(
    [sys.executable, "-u", str(repo / "scripts/colab.py")],
    cwd=repo,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError("起動に失敗しました。上のログを確認してください。")
del env